# 📋 Rapport d'Avancement du Projet MLOps — ImmoPrix

**Objectif du Projet :** Développer, déployer et monitorer un modèle de Machine Learning capable de prédire le prix médian des maisons en Californie (`MedHouseVal`) pour accompagner les agents immobiliers d'ImmoPrix.

---

## 🔍 Mission 1 : Exploration et Préparation des Données ( DuckDB )

### Actions Réalisées :
* **Exploration :** EDA (describe, histos, heatmap, scatter géo) + check NaN (aucune) + outliers IQR (conservés)
* **Ingestion :** Extraction du dataset *California Housing* et centralisation dans une base de données locale performante : `data/processed/housing.duckdb`.
* **Automatisation :** Création d'un script de nettoyage reproductible (`prepare.py`) piloté par le `Makefile` (`make prepare`).

---

## ⚙️ Mission 2 : Modélisation et Tracking des Expériences ( MLflow )

Trois architectures ont été entraînées sur les données processées et scalées (StandardScaler), puis trackées au sein de notre serveur MLflow local.

### 📊 Tableau Comparatif des Performances :

| Modèle | Type | R² Score (Test) | RMSE | Statut |
| :--- | :--- | :---: | :---: | :---: |
| **Baseline** | Régression Linéaire | ~ 0.60 | ~ 0.75 | Écarté |
| **Gradient Boosting** | Ensembles (Boosting) | 0.73 | ~ 0.58 | Écarté |
| **Random Forest** | Ensembles (Bagging) | **0.76** | **0.55** | **Champion 🏆** |

### Conclusions MLOps :
1. Les modèles non-linéaires écrasent la baseline, confirmant la complexité des relations géographiques et économiques des données.
2. Le **Random Forest** obtient les meilleures performances ($R^2 = 0.76$). Il a été enregistré dans le **Model Registry** de MLflow sous le nom de `ImmoPrix_Champion` pour servir de point d'ancrage à la future API de production.

---

## 🕵️‍♂️ Mission 3 : Analyse des Features et Explicabilité ( SHAP )

Nous avons ouvert la "boîte noire" de notre modèle Champion (Random Forest) grâce à l'approche théorique des valeurs de Shapley (XAI) afin d'assurer la transparence métier pour les agents d'ImmoPrix.

### 1. Analyse Globale (Importance des variables)
* **Le facteur Roi :** `MedInc` (le revenu médian du quartier) est, de très loin, la variable la plus prédictive. Un revenu élevé augmente drastiquement le prix de l'immobilier.
* **La Géographie :** La localisation (`Latitude` / `Longitude`) représente le second levier d'impact. Le modèle capte parfaitement les disparités de prix entre les côtes ultra-prisées (Sud/San Francisco) et l'intérieur des terres.
* **L'infrastructure :** L'âge de la maison ou le nombre de pièces (`AveRooms`) n'ont qu'un impact marginal face aux critères socio-économiques et géographiques (*"Emplacement, Emplacement, Emplacement"*).

### 2. Analyse Locale (Waterfall Plot sur un exemple)
L'analyse d'une prédiction individuelle a permis de valider le comportement logique du modèle :
* **Cas d'étude :** Une maison estimée à **$141,100** (en dessous de la moyenne globale de $195,600).
* **Explication :** Bien que son âge avancé (+9k$) lui donne de la valeur, la précarité économique du secteur (`MedInc` très bas) retire à elle seule près de -51k$, expliquant la décote finale calculée par le modèle.

---

## 🚀 Prochaines Étapes (Mission 4)
* Développer une API d'inférence temps réel avec **FastAPI** capable de charger le modèle `ImmoPrix_Champion`.
* Conteneuriser l'application avec **Docker** pour garantir sa portabilité.
* Concevoir une interface graphique interactive avec **Streamlit** intégrant ces visuels SHAP pour les utilisateurs finaux.